For this project, I am using run7722all_metafix.root that was produced by Arran using his custom Data processor that includes the muon hits (muon_paddle_hits) and the positions of each paddle in the meta tree.<br>

The data contains the data processed for the water PMTs also to link the muon hits with the PMT pulses.<br>


In [ ]:
# This block install some 3D plotting capability to your python install. 
# Uncomment and run once to install ipymp or plotly
#%pip install ipympl
#%pip install plotly

In [ ]:
# Let's print out the output tree.
import ROOT
f = ROOT.TFile.Open("../Data/run7722all_metafix.root")
output = f.Get("output")
output.Print()

The print out shows that the output tree has 353719 events. The useful variables for this analysis are
- muon_paddle_hits: 1s and 0s whether or not the paddle is hit out of the 134 total.
- digitNHits or digitNhitsCleaned: This is tell us how much energy is deposited in the water tank.
- digitCharge: Same as above but more direct correlation to energy.

In [ ]:
# This block just check if the digitNhits are valid for this dataset.
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
# Create RDataFrame from the events tree of the input file.
rdf = ROOT.RDataFrame("output", "../Data/run7722all_metafix.root")
# Extract nhits values
digitNhits = rdf.AsNumpy(columns=["digitNhits"])["digitNhits"]
# Histogramming
digitNhits_counts, digitNhits_bins = np.histogram(digitNhits, bins=np.arange(0, 240, 1))
digitNhits_bin_centers = [(digitNhits_bins[i] + digitNhits_bins[i + 1]) / 2 for i in range(len(digitNhits_bins) - 1)]
# Plotting
plt.figure(figsize=(12, 6))
plt.title(f"nhits histogram")
plt.errorbar(digitNhits_bin_centers, digitNhits_counts,
             ls='', marker='o', mfc='black', ms=4, mec='black',
             ecolor='black', label="Run 7722")

plt.xlabel("digitNhits", fontsize=18)
plt.ylabel("NEvents", fontsize=15)
plt.yscale('log')
#plt.ylim(3000,250_000)
plt.legend()
plt.show()

This digitNhtis looks pretty normal for Muon Data. There's probably a lot of different events and noise mix into it but hopefully the Muon paddle data will help with this.

In [ ]:
# This prints out the meta tree.
import ROOT
f = ROOT.TFile.Open("../Data/run7722all_metafix.root")
meta = f.Get("meta")
meta.Print()

We need the values from muon_paddle_x,y,z.

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

meta = ROOT.RDataFrame("meta", "../Data/run7722all_metafix.root")

#muon_paddle_x = meta.AsNumpy(columns=["muon_paddle_x"])
#print(muon_paddle_x)

#muon_paddle_y = meta.AsNumpy(columns=["muon_paddle_y"])
#print(muon_paddle_y)

muon_paddle_z = meta.AsNumpy(columns=["muon_paddle_z"])["muon_paddle_z"][0]
print(muon_paddle_z.shape)
#print(muon_paddle_z[0].shape)

muon_paddle_x,y,z seems to be 164x134 shape. The 134 is for the number of paddles, 164 is actually from the number of root/hdf5 files that were combined together.

What I want to do is to combine the position in the meta and hits from output together into a single array for analysis.

## Run this box to produce muon_pos_hits array

In [1]:
# Code to combine layer#, xyz positions of the paddle and the event by event hits.
# Layer is the row 0, xyz are row 1 to 3, hit data are 4 to NEvents+4
import ROOT
import numpy as np
import matplotlib.pyplot as plt

meta = ROOT.RDataFrame("meta", "../Data/run7722all_metafix.root")
# Get the xyz position, but we just need the 1st entry (hdf5 file).
x_pos = meta.AsNumpy(columns=["muon_paddle_x"])["muon_paddle_x"][0]
y_pos = meta.AsNumpy(columns=["muon_paddle_y"])["muon_paddle_y"][0]
z_pos = meta.AsNumpy(columns=["muon_paddle_z"])["muon_paddle_z"][0]
#print("x_pos shape:", x_pos.shape)
#print("y_pos shape:", y_pos.shape)
#print("z_pos shape:", z_pos.shape)

output = ROOT.RDataFrame("output", "../Data/run7722all_xyz.root")
# Get the muon_paddle_hit from the output tree.
# Convert from object array of event vectors to 2D array: NEvents x 134
muon_hits = np.stack(output.AsNumpy(["muon_paddle_hit"])["muon_paddle_hit"])

print("muon_hits shape:", muon_hits.shape)

# Put a value for the layers 
layer = np.zeros_like(z_pos, dtype=int) # defaults to 0
layer[np.isclose(z_pos, 2129.984)] = 1 # TopUpper-x
layer[np.isclose(z_pos, 2098.311)] = 2 # TopUpper+x
layer[np.isclose(z_pos, 1972.377)] = 3 # TopLower-y
layer[np.isclose(z_pos, 1940.704)] = 4 # TopLower+y
layer[np.isclose(z_pos, 699.896)] = 5 # Barrel+z
layer[np.isclose(z_pos, -398.908)] = 6 # Barrel-z
layer[np.isclose(z_pos, -1807.338)] = 7 # Bottom-y
layer[np.isclose(z_pos, -1839.012)] = 8 # Bottom+y

# Stack x, y, z as first 3 rows
muon_pos_hits = np.vstack([
    layer,
    x_pos,
    y_pos,
    z_pos,
    muon_hits
])

print("muon_hits_with_pos shape:", muon_pos_hits[0])

muon_hits shape: (353719, 134)
muon_hits_with_pos shape: [6. 5. 6. 3. 6. 5. 3. 4. 6. 5. 0. 0. 8. 2. 2. 3. 0. 8. 8. 8. 8. 8. 6. 6.
 6. 6. 6. 8. 8. 8. 8. 8. 6. 6. 6. 6. 6. 6. 6. 6. 4. 5. 4. 4. 3. 4. 3. 7.
 1. 1. 4. 4. 7. 7. 7. 7. 7. 5. 5. 5. 5. 5. 7. 7. 7. 7. 7. 5. 5. 5. 5. 5.
 5. 5. 5. 6. 0. 3. 0. 3. 4. 0. 6. 6. 5. 5. 6. 6. 5. 5. 6. 6. 5. 5. 6. 6.
 5. 5. 6. 6. 5. 5. 6. 6. 5. 5. 0. 0. 2. 6. 1. 5. 2. 6. 1. 5. 2. 2. 1. 1.
 2. 2. 1. 1. 2. 2. 1. 1. 3. 6. 4. 5. 0. 0.]


In [ ]:
# Just checking the xyz for particular z position paddles
print(muon_pos_hits[1:4,np.where(np.isclose(muon_pos_hits[3,], -1839.012))[0]])

## Let's look for the easiest events first.
The muon passes through TopUpper, TopLower and Bottom. Assuming the muon is travel almost straight, the three paddle should have a path that can be intersected by a straight line.<br>

Let's filter for the events that has 1 hit on each TopUpper, TopLower and Bottom and no hits on the barrel.

In [2]:
# Get columns from layer info
TopUpper = np.where((muon_pos_hits[0] == 1) | (muon_pos_hits[0] == 2))[0]
TopLower = np.where((muon_pos_hits[0] == 3) | (muon_pos_hits[0] == 4))[0]
Bottom = np.where((muon_pos_hits[0] == 7) | (muon_pos_hits[0] == 8))[0]
Barrel = np.where((muon_pos_hits[0] == 5) | (muon_pos_hits[0] == 6))[0]

#print("TopUpper columns:", TopUpper)
#print("TopLower columns:", TopLower)

# Count hits only in the selected layer groups
top_upper_hits = np.count_nonzero(muon_pos_hits[4:, TopUpper] == 1, axis=1)
top_lower_hits = np.count_nonzero(muon_pos_hits[4:, TopLower] == 1, axis=1)
bottom_hits = np.count_nonzero(muon_pos_hits[4:, Bottom] == 1, axis=1)
barrel_hits = np.count_nonzero(muon_pos_hits[4:, Barrel] == 1, axis=1)
event_list = []

for event_number, upper_count, lower_count, bottom_count, barrel_count in zip(
    range(len(muon_pos_hits[4:])),
    top_upper_hits,
    top_lower_hits,
    bottom_hits,
    barrel_hits
):
    if upper_count == 1 and lower_count == 1 and bottom_count == 1 and barrel_count == 0:
        event_list.append(event_number)
        print(
            f"Event {event_number}: "
            f"TopUpper = {upper_count} hits, "
            f"TopLower = {lower_count} hits"
            f"Bottom = {bottom_count} hits"
            f"Barrel = {barrel_count} hits"
        
        )
print("len(event_list), ",len(event_list))

Event 38: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 331: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 526: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 677: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 834: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 856: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1044: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1050: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1099: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1214: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1456: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1733: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 hitsBarrel = 0 hits
Event 1775: TopUpper = 1 hits, TopLower = 1 hitsBottom = 1 

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from MuonTrack import *
# Use Qt backend for smoother 3D rotation
# Run this once in a separate notebook cell if needed:
%matplotlib qt

EVENTNUM = 38


event_hit_row = 4 + EVENTNUM

hit_columns = np.where(muon_pos_hits[event_hit_row] == 1)[0]

print(f"Event {EVENTNUM}")
print("hit columns:", hit_columns)
print("number of hits:", len(hit_columns))

fig = plt.figure(figsize=(10, 11))
ax = fig.add_subplot(111, projection="3d")

for col in hit_columns:
    layer_value = int(muon_pos_hits[0, col])

    x = muon_pos_hits[1, col]
    y = muon_pos_hits[2, col]
    z = muon_pos_hits[3, col]

    print(f"  col {col}: layer={layer_value}, x={x}, y={y}, z={z}")

    if layer_value == 1:
        draw_paddle(ax, center=(x, y, z), size=(1300, 210, 10), alpha=0.35,facecolor=color_from_layer(layer_value))
    elif layer_value == 2:
        draw_paddle(ax, center=(x, y, z), size=(1300, 210, 10), alpha=0.35,facecolor=color_from_layer(layer_value))
    elif layer_value == 3:
        draw_paddle(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35,facecolor=color_from_layer(layer_value))    
    elif layer_value == 4:
        draw_paddle(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35,facecolor=color_from_layer(layer_value))
    elif layer_value == 5:
        draw_paddle(ax, center=(x, y, z), size=(210, 210, 1300), alpha=0.35,facecolor=color_from_layer(layer_value))
    elif layer_value == 6:
        draw_paddle(ax, center=(x, y, z), size=(210, 210, 1300), alpha=0.35,facecolor=color_from_layer(layer_value))
    elif layer_value == 7:
        draw_paddle(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35,facecolor=color_from_layer(layer_value))    
    elif layer_value == 8:
        draw_paddle(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35,facecolor=color_from_layer(layer_value))
    else:
        ax.scatter(x, y, z, s=80)

    ax.text(x, y, z, layer_value, fontsize=9)


ax.set_xlabel("x_pos")
ax.set_ylabel("y_pos")
ax.set_zlabel("z_pos")
ax.set_title(f"Event {EVENTNUM}: Muon Hits with Layer #")

ax.set_xlim(-1600, 1600)
ax.set_ylim(-1600, 1600)
ax.set_zlim(-2200, 2200)

ax.set_box_aspect((3200, 3200, 4400))

plt.show()

Event 38
hit columns: [ 46  65 122]
number of hits: 3
  col 46: layer=3, x=488.95, y=-549.402, z=1972.377
  col 65: layer=7, x=-508.0, y=-549.402, z=-1807.338
  col 122: layer=1, x=-549.402, y=-488.95, z=2129.984


## Now let's try to fit tracks through these boxes if possible.
The best track should be those going through the center of each box.

In [4]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from MuonTrack import *
# Use Qt backend for smoother 3D rotation
# Run this once in a separate notebook cell if needed:
%matplotlib qt

EVENTNUM = 346216

event_hit_row = 4 + EVENTNUM

hit_columns = np.where(muon_pos_hits[event_hit_row] == 1)[0]

print(f"Event {EVENTNUM}")
print("hit columns:", hit_columns)
print("number of hits:", len(hit_columns))

fig = plt.figure(figsize=(10, 11))
ax = fig.add_subplot(111, projection="3d")
hit_boxes = []
hit_centers = []
hit_layers = []
hit_cols = []


for col in hit_columns:
    layer_value = int(muon_pos_hits[0, col])

    x = muon_pos_hits[1, col]
    y = muon_pos_hits[2, col]
    z = muon_pos_hits[3, col]

    print(f"  col {col}: layer={layer_value}, x={x}, y={y}, z={z}")
    
    size = box_size_from_layer(layer_value)
    
    if size is not None:
        draw_paddle(ax, center=(x, y, z), size=size, alpha=0.35, facecolor=color_from_layer(layer_value))

    ax.text(x, y, z, str(layer_value), fontsize=9)

    if size is not None:
        hit_boxes.append({
            "center": np.array([x, y, z], dtype=float),
            "size": np.array(size, dtype=float),
            "bounds": box_bounds(center=(x, y, z), size=size),
            "col": col,
            "layer": layer_value,
        })
    
    hit_centers.append([x, y, z])
    hit_layers.append(layer_value)
    hit_cols.append(col)


ax.set_xlabel("x_pos")
ax.set_ylabel("y_pos")
ax.set_zlabel("z_pos")
ax.set_title(f"Event {EVENTNUM}: Muon Hits with Layer #")

ax.set_xlim(-1600, 1600)
ax.set_ylim(-1600, 1600)
ax.set_zlim(-2200, 2200)

ax.set_box_aspect((3200, 3200, 4400))
hit_centers = np.array(hit_centers, dtype=float)

if len(hit_boxes) >= 2:
    line_point, line_dir, score = fit_line_through_boxes(
        hit_boxes,
        n_trials=10000,
        seed=12345
    )

    if line_point is None:
        print("No line found that intersects all hit boxes.")
        print("Try increasing n_trials.")
    else:
        # Project box centers onto fitted line to choose display length
        centers = np.array([b["center"] for b in hit_boxes], dtype=float)
        t_values = (centers - line_point) @ line_dir

        t_min = t_values.min() - 700
        t_max = t_values.max() + 700

        t_line = np.linspace(t_min, t_max, 100)
        line_xyz = line_point + np.outer(t_line, line_dir)

        ax.plot(
            line_xyz[:, 0],
            line_xyz[:, 1],
            line_xyz[:, 2],
            color="red",
            linewidth=3,
            label="box-aware central line"
        )

        print("\nBox-aware fitted line:")
        print("  point     =", line_point)
        print("  direction =", line_dir)
        print("  score     =", score)

        print("\nDistances from hit box centers to fitted line:")
        for box in hit_boxes:
            dist = point_line_distance(box["center"], line_point, line_dir)
            print(
                f"  col {box['col']}, layer {box['layer']}: "
                f"distance = {dist:.3f}"
            )

        ax.legend()
else:
    print("Need at least 2 hit boxes to fit a line.")
    
plt.show()

Event 346216
hit columns: [ 46  65 121]
number of hits: 3
  col 46: layer=3, x=488.95, y=-549.402, z=1972.377
  col 65: layer=7, x=-508.0, y=-549.402, z=-1807.338
  col 121: layer=2, x=549.402, y=488.95, z=2098.311
No line found that intersects all hit boxes.
Try increasing n_trials.


In [5]:
# Get columns from layer info
TopUpper = np.where((muon_pos_hits[0] == 1) | (muon_pos_hits[0] == 2))[0]
TopLower = np.where((muon_pos_hits[0] == 3) | (muon_pos_hits[0] == 4))[0]
Bottom = np.where((muon_pos_hits[0] == 7) | (muon_pos_hits[0] == 8))[0]
Barrel = np.where((muon_pos_hits[0] == 5) | (muon_pos_hits[0] == 6))[0]

#print("TopUpper columns:", TopUpper)
#print("TopLower columns:", TopLower)

# Count hits only in the selected layer groups
top_upper_hits = np.count_nonzero(muon_pos_hits[4:, TopUpper] == 1, axis=1)
top_lower_hits = np.count_nonzero(muon_pos_hits[4:, TopLower] == 1, axis=1)
bottom_hits = np.count_nonzero(muon_pos_hits[4:, Bottom] == 1, axis=1)
barrel_hits = np.count_nonzero(muon_pos_hits[4:, Barrel] == 1, axis=1)
event_list = []

for event_number, upper_count, lower_count, bottom_count, barrel_count in zip(
    range(len(muon_pos_hits[4:])),
    top_upper_hits,
    top_lower_hits,
    bottom_hits,
    barrel_hits
):
    if upper_count == 1 and lower_count == 1 and bottom_count == 1 and barrel_count == 0:
        event_list.append(event_number)
       # print(
       #     f"Event {event_number}: "
       #     f"TopUpper = {upper_count} hits, "
       #     f"TopLower = {lower_count} hits"
       #     f"Bottom = {bottom_count} hits"
       #     f"Barrel = {barrel_count} hits"
       #     )

fit_results = []

for EVENTNUM in event_list:
    result = test_event_fit(
        EVENTNUM,
        muon_pos_hits,
        n_trials=10000,
        seed=12345
    )

    fit_results.append(result)

    status = "SUCCESS" if result["success"] else "FAIL"

    print(
        f"Event {EVENTNUM}: {status}, "
        f"n_hits={result['n_hits']}, "
        f"score={result['score']}, "
        f"reason={result['reason']}"
    )

    print(f"  hit columns: {result['hit_columns']}")

    if result["success"]:
        print(f"  line point: {result['line_point']}")
        print(f"  line dir  : {result['line_dir']}")

    print()

n_success = sum(1 for r in fit_results if r["success"])
n_fail = sum(1 for r in fit_results if not r["success"])

print(f"Total success: {n_success}")
print(f"Total fail:    {n_fail}")
print(f"Total events:  {len(fit_results)}")

Event 38: FAIL, n_hits=3, score=None, reason=no line found that intersects all hit boxes
  hit columns: [46, 65, 122]

Event 331: FAIL, n_hits=3, score=None, reason=no line found that intersects all hit boxes
  hit columns: [65, 79, 119]

Event 526: FAIL, n_hits=3, score=None, reason=no line found that intersects all hit boxes
  hit columns: [65, 79, 119]

Event 677: SUCCESS, n_hits=3, score=0.23610328421159427, reason=fit successful
  hit columns: [3, 65, 126]
  line point: [ -508.17516516  -537.06828526 -1809.87346243]
  line dir  : [-0.04865774  0.06704948  0.99656249]

Event 834: FAIL, n_hits=3, score=None, reason=no line found that intersects all hit boxes
  hit columns: [43, 65, 76, 119]

Event 856: FAIL, n_hits=3, score=None, reason=no line found that intersects all hit boxes
  hit columns: [46, 65, 126]

Event 1044: FAIL, n_hits=3, score=None, reason=no line found that intersects all hit boxes
  hit columns: [46, 65, 119]

Event 1050: FAIL, n_hits=3, score=None, reason=no line 

## Looking at DigiNhits for events that has a straight track fitting or not.

In [12]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

rdf = ROOT.RDataFrame("output", "../Data/run7722all_metafix.root")

digitNhits = rdf.AsNumpy(columns=["digitNhits"])["digitNhits"]
n_events = len(digitNhits)

bins = np.arange(0, 240, 5)

success_counts, digitNhits_bins = np.histogram(digitNhits[np.array(
    [r["event"] for r in fit_results if r["success"]],
    dtype=int
)], bins=bins)

fail_counts, _ = np.histogram(digitNhits[np.array([r["event"] for r in fit_results if not r["success"]],dtype=int)], bins=bins)

# Normalize fail histogram to same total as success histogram
if np.sum(fail_counts) > 0:
    fail_counts = fail_counts * np.sum(success_counts) / np.sum(fail_counts)

digitNhits_bin_centers = 0.5 * (digitNhits_bins[:-1] + digitNhits_bins[1:])

plt.figure(figsize=(12, 6))
plt.title("digitNhits histogram: fit success vs fit fail")

plt.plot(
    digitNhits_bin_centers,
    success_counts,
    color="blue",
    label=f"Fit success ({len(event_success)} events)"
)

plt.plot(
    digitNhits_bin_centers,
    fail_counts,
    color="red",
    label=f"Fit fail normalized ({len(event_fail)} events)"
)

plt.xlabel("digitNhits", fontsize=18)
plt.ylabel("NEvents", fontsize=15)
plt.yscale("log")
plt.legend()
plt.show()

In [28]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

def plot_digitNhits_event_list(
    event_list,
    root_file="../Data/run7722all_metafix.root",
    tree_name="output",
    bins=np.arange(0, 210, 5),
    normalize_red=True
):
    rdf = ROOT.RDataFrame(tree_name, root_file)

    digitNhits = rdf.AsNumpy(columns=["digitNhits"])["digitNhits"]
    n_events = len(digitNhits)

    event_list = np.array(event_list, dtype=int)

    # Keep only valid event numbers
    event_list = event_list[(event_list >= 0) & (event_list < n_events)]

    # Blue: events in event_list
    selected_events = event_list

    # Red: events not in event_list
    all_events = np.arange(n_events, dtype=int)
    other_events = np.setdiff1d(all_events, selected_events)

    selected_digitNhits = digitNhits[selected_events]
    other_digitNhits = digitNhits[other_events]

    selected_counts, digitNhits_bins = np.histogram(
        selected_digitNhits,
        bins=bins
    )

    other_counts, _ = np.histogram(
        other_digitNhits,
        bins=bins
    )

    # Optional: normalize red histogram to same total as blue histogram
    if normalize_red and np.sum(other_counts) > 0:
        other_counts = other_counts * np.sum(selected_counts) / np.sum(other_counts)

    digitNhits_bin_centers = 0.5 * (
        digitNhits_bins[:-1] + digitNhits_bins[1:]
    )

    plt.figure(figsize=(12, 6))
    plt.title("digitNhits histogram: selected events vs all other events")

    plt.errorbar(
        digitNhits_bin_centers[1:-1],
        selected_counts[1:-1],
        yerr=np.sqrt(selected_counts[1:-1]),
        color="black",
        marker=".",
        linestyle="",
        capsize=2,
        label=f"Triple Hits ({len(selected_events)} events)"
    )

    plt.bar(
        digitNhits_bin_centers[1:-1],
        other_counts[1:-1],
        width=5,
        color="blue",
        alpha=0.5,
        label=f"Not Triple Hits ({len(other_events)} events)"
    )

    plt.xlabel("digitNhits", fontsize=18)
    plt.ylabel("NEvents", fontsize=15)
    #plt.yscale("log")
    plt.legend()
    plt.show()

    return selected_events, other_events, selected_counts, other_counts

selected_events, other_events, selected_counts, other_counts = plot_digitNhits_event_list(
    event_list,
    normalize_red=True
)

In [6]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

def plot_digitTotalCharge_event_list(
    event_list,
    root_file="../Data/run7722all_metafix.root",
    tree_name="output",
    charge_leaf="digitCharge",
    bins=None,
    normalize_red=True
):
    import matplotlib.pyplot as plt
    from cycler import cycler

    eos_colors = [
        "violet",
        "royalblue",
        "springgreen",
        "forestgreen",
        "darkkhaki",
        "gold",
        "orangered",
        "brown",
    ]

    plt.rcParams["axes.ymargin"] = 0.05
    plt.rcParams["axes.xmargin"] = 0.05
    plt.rcParams["axes.prop_cycle"] = cycler(color=eos_colors)
    plt.rcParams["legend.fontsize"] = "x-large"
    plt.rcParams["legend.framealpha"] = 0
    plt.rcParams["legend.title_fontsize"] = "x-large"
    rdf = ROOT.RDataFrame(tree_name, root_file)

    # Read charge array branch
    digitCharge = rdf.AsNumpy(columns=[charge_leaf])[charge_leaf]

 # Sum only charges above 0.5 in each event
    digitTotalCharge = np.array(
        [np.sum(np.asarray(charges)[np.asarray(charges) > 0.5]) for charges in digitCharge],
        dtype=float
    )

    n_events = len(digitTotalCharge)

    event_list = np.array(event_list, dtype=int)
    event_list = event_list[(event_list >= 0) & (event_list < n_events)]

    selected_events = event_list
    all_events = np.arange(n_events, dtype=int)
    other_events = np.setdiff1d(all_events, selected_events)

    selected_charge = digitTotalCharge[selected_events]
    other_charge = digitTotalCharge[other_events]


    selected_counts, charge_bins = np.histogram(
        selected_charge,
        bins=np.arange(0,13000,260)
    )

    other_counts, _ = np.histogram(
        other_charge,
        bins=charge_bins
    )

    # Normalize red histogram to same total as blue histogram
    if normalize_red and np.sum(other_counts) > 0:
        other_counts = other_counts * np.sum(selected_counts) / np.sum(other_counts)

    charge_bin_centers = 0.5 * (
        charge_bins[:-1] + charge_bins[1:]
    )

    fig, ax = plt.subplots(figsize=(10, 9))

    ax.set_title("Total Integrated charge (pC) pass-through events vs all other events")

    ax.bar(
        charge_bin_centers,
        other_counts,
        width=260,
        color="royalblue",
        alpha=0.5,
        label=f"Not Triple Hits ({len(other_events)} events)"
    )

    ax.errorbar(
        charge_bin_centers,
        selected_counts,
        yerr=np.sqrt(selected_counts),
        color="black",
        marker=".",
        linestyle="",
        capsize=2,
        label=f"Triple Hits ({len(selected_events)} events)"
    )

    ax.set_xlabel("Total Integrated charge (pC)", fontsize=14)
    ax.set_ylabel("Arb. Units", fontsize=14)
    ax.set_yscale("log")

    ax.legend(framealpha=0)

    plt.show()

    return selected_events, other_events, digitTotalCharge, selected_counts, other_counts

selected_events, other_events, digitTotalCharge, selected_counts, other_counts = (
    plot_digitTotalCharge_event_list(event_list)
)

qt.qpa.wayland: Ignoring unexpected wl_surface.leave received for output with id: 33 screen name: "rdp-2" screen model: "rdp" This is most likely a bug in the compositor.


In [ ]:
total_counts = success_counts + fail_counts

# Avoid divide-by-zero where both are zero
fail_fraction = np.divide(
    fail_counts,
    total_counts,
    out=np.zeros_like(fail_counts, dtype=float),
    where=total_counts > 0
)

bad_cols = col_bin_centers[fail_fraction > 0.75].astype(int)

print("Columns with fail fraction > 75%:")
print(bad_cols.tolist())

print("\nDetailed:")
for col, frac, fail, success in zip(
    col_bin_centers.astype(int),
    fail_fraction,
    fail_counts,
    success_counts
):
    if frac > 0.75:
        print(
            f"column {col}: "
            f"fail_fraction = {100*frac:.1f}%, "
            f"fail = {fail:.2f}, "
            f"success = {success:.2f}"
        )

In [ ]:
success_mask = success_counts > 0
fail_mask = fail_counts > 0

union_mask = success_mask | fail_mask
overlap_mask = success_mask & fail_mask
success_only_mask = success_mask & ~fail_mask
fail_only_mask = fail_mask & ~success_mask

print(f"Success columns:      {np.count_nonzero(success_mask)}")
print(f"Fail columns:         {np.count_nonzero(fail_mask)}")
print(f"Union columns:        {np.count_nonzero(union_mask)}")
print(f"Overlap columns:      {np.count_nonzero(overlap_mask)}")
print(f"Success-only columns: {np.count_nonzero(success_only_mask)}")
print(f"Fail-only columns:    {np.count_nonzero(fail_only_mask)}")

In [ ]:
print(muon_pos_hits[0:4,80])

In [14]:
%load_ext autoreload
%autoreload 2
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from MuonTrack import *

fig = plt.figure(figsize=(12, 11))
ax = fig.add_subplot(111, projection="3d")

n_paddles = muon_pos_hits.shape[1]

for col in range(n_paddles):
    layer_value = int(muon_pos_hits[0, col])

    x = muon_pos_hits[1, col]
    y = muon_pos_hits[2, col]
    z = muon_pos_hits[3, col]

    size = box_size_from_layer(layer_value)

    if size is None:
        print(f"Skipping col {col}: unknown layer {layer_value}")
        continue

    if layer_value <= 0:
        continue

    if layer_value in [5, 6]:
        # Barrel paddle:
        # 210 wide tangentially, 10 thick radially, 1300 long in z
        radial_axis, tangential_axis, z_axis = barrel_axes_from_position(x, y)

        draw_oriented_box(
            ax,
            center=(x, y, z),
            axes=(tangential_axis, radial_axis, z_axis),
            size=(210, 10, 1300),
            alpha=0.15,
            facecolor=color_from_layer(layer_value)
        )

    else:
        # Non-barrel paddles remain axis-aligned
        draw_paddle(
            ax,
            center=(x, y, z),
            size=size,
            alpha=0.15,
            facecolor=color_from_layer(layer_value)
        )

    ax.text(x, y, z, str(col), fontsize=7)

ax.set_xlabel("x_pos")
ax.set_ylabel("y_pos")
ax.set_zlabel("z_pos")
ax.set_title("All Muon Paddles")

ax.set_xlim(-1600, 1600)
ax.set_ylim(-1600, 1600)
ax.set_zlim(-2200, 2200)

ax.set_box_aspect((3200, 3200, 4400))

plt.show()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Skipping col 10: unknown layer 0
Skipping col 11: unknown layer 0
Skipping col 51: unknown layer 0
Skipping col 78: unknown layer 0
Skipping col 81: unknown layer 0
Skipping col 106: unknown layer 0
Skipping col 107: unknown layer 0
Skipping col 132: unknown layer 0
Skipping col 133: unknown layer 0
